# Email Spam Detection with Machine Learning
**Author:** Syed Aon Muhammad Kazmi  
**Track:** Data Science | OASIS INFOBYTE

## Objective and data
Data: UCI SMS Spam Collection, https://archive.ics.uci.edu/dataset/228/sms+spam+collection. The assignment allows this dataset, but it contains SMS messages, not emails. Results therefore support an SMS spam classifier, not validated email performance.

Reproducible seed: 42. Model selection uses cross-validation on training data; the held-out test set is evaluated once per candidate for reporting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             mean_absolute_error, mean_squared_error, r2_score)
sns.set_theme(style='whitegrid', palette='colorblind')
SEED = 42
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
def save(name):
    plt.savefig(OUT / f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show()
def inspect(df):
    print('Shape:', df.shape)
    display(df.dtypes.to_frame('dtype'), df.isna().sum().to_frame('missing'), df.drop(columns=['text','clean'], errors='ignore').describe(include='all'))


In [ ]:
import zipfile,re,urllib.request
# Obtain the public UCI dataset locally; raw messages are not published in this repository.
DATA = Path('data/spam.zip'); DATA.parent.mkdir(exist_ok=True)
if not DATA.exists():
    urllib.request.urlretrieve('https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip', DATA)
with zipfile.ZipFile('data/spam.zip') as z:
    df=pd.read_csv(z.open('SMSSpamCollection'),sep='\t',header=None,names=['label','text'])
inspect(df)
distribution=df.label.value_counts().to_frame('count');distribution['percent']=distribution['count']/len(df)*100;display(distribution)
sns.countplot(data=df,x='label');save('class_distribution')
# Remove all examples with contradictory labels, then duplicate texts.
conflicts=df.groupby('text').label.nunique();bad=conflicts[conflicts>1].index
df=df[~df.text.isin(bad)].drop_duplicates('text').copy()
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS,TfidfVectorizer
def clean_text(value):
    tokens=re.sub(r'[^a-z0-9\s]',' ',value.lower()).split()
    return ' '.join(t for t in tokens if t not in ENGLISH_STOP_WORDS)
df['clean']=df.text.map(clean_text)
# Prevent identical normalised messages crossing train/test.
conflicts=df.groupby('clean').label.nunique();df=df[~df.clean.isin(conflicts[conflicts>1].index)].drop_duplicates('clean')
print('Clean shape:',df.shape)

## TF-IDF and preprocessing
Term frequency measures a word’s frequency within a message. Inverse document frequency downweights words found across many messages. Their product highlights informative terms. We lowercase, remove punctuation and English stopwords; stemming is omitted to preserve tokens. This simple treatment can lose useful punctuation cues. Vocabulary and IDF are fitted only within training folds.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
X_train,X_test,y_train,y_test=train_test_split(df.clean,df.label,test_size=.2,stratify=df.label,random_state=SEED)
models={'Naive Bayes':make_pipeline(TfidfVectorizer(ngram_range=(1,2)),MultinomialNB(alpha=.5)), 'Logistic Regression':make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=1000,class_weight='balanced',random_state=SEED))}
from sklearn.metrics import make_scorer,f1_score
cv=StratifiedKFold(5,shuffle=True,random_state=SEED);rows=[]
for name,model in models.items():
    score=cross_val_score(model,X_train,y_train,cv=cv,scoring=make_scorer(f1_score,pos_label='spam')).mean()
    model.fit(X_train,y_train);pred=model.predict(X_test)
    precision,recall,f1,_=precision_recall_fscore_support(y_test,pred,average='binary',pos_label='spam',zero_division=0)
    rows.append({'Model':name,'CV spam F1':score,'Accuracy':accuracy_score(y_test,pred),'Spam precision':precision,'Spam recall':recall,'Spam F1':f1})
    print(name);print(classification_report(y_test,pred,zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['ham','spam'],labels=['ham','spam']);save(name.replace(' ','_')+'_confusion')
results=pd.DataFrame(rows).sort_values('CV spam F1',ascending=False);display(results);results.to_csv(OUT/'metrics.csv',index=False)
best_name=results.iloc[0]['Model'];print('Selected:',best_name,'by training CV spam F1, balancing precision and recall.')
# Apply the fitted classifier to your own local messages when needed.

## Why recall matters
Spam recall measures how much real spam is caught. Low recall lets unwanted or dangerous messages through. Precision also matters: incorrectly blocking legitimate messages can be costly. Accuracy alone is misleading with more ham than spam. Selection uses spam F1 as a balanced baseline; a production threshold should reflect user costs and be tuned on validation data.

## Limits
This older SMS dataset cannot establish performance on modern emails, multilingual messages or evolving scams. The examples above are demonstrations, not an independent evaluation.

## Results from the verified run
Logistic Regression wins training CV spam F1 (0.908) and reaches held-out spam F1 0.902, precision 0.891 and recall 0.913. Naive Bayes has higher precision (0.989) but lower recall (0.730). The selected model catches more spam at the cost of more false positives.